# Run Local Shopping Chatbot

This notebook runs a local shopping chatbot with:

- Base model: `Qwen/Qwen2.5-3B-Instruct`
- Trained LoRA adapter: `models/shopping_chatbot_lora_adapter`
- Product data: `processed/catalog.csv`

The adapter is not a full model. The first run may download the base model from Hugging Face.


## Install Dependencies

Run this cell if the inference dependencies are not installed.


In [ ]:
# %pip install -U torch transformers accelerate peft sentencepiece pandas
# If you see `operator torchvision::nms does not exist`, run:
# %pip uninstall -y torchvision


## Configuration


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/Users/huybui/Desktop/Smart Shopping Chatbot")
PRODUCTS_PATH = PROJECT_ROOT / "processed" / "catalog.csv"
ADAPTER_PATH = PROJECT_ROOT / "models" / "shopping_chatbot_lora_adapter"

BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # Use only if the adapter was also trained from the 1.5B base model.

TOP_K_PRODUCTS = 5
MAX_CONTEXT_CHARS = 4500

print("Products:", PRODUCTS_PATH)
print("Adapter:", ADAPTER_PATH)
print("Adapter exists:", ADAPTER_PATH.exists())


## Load Product Data


In [ ]:
import re
import pandas as pd

products = pd.read_csv(PRODUCTS_PATH).fillna("")
print("Products:", len(products))
products.head()


## Product Retrieval

This section uses simple keyword matching to retrieve relevant products. It can later be replaced with embeddings or a vector database.


In [ ]:
CATEGORY_KEYWORDS = {
    "điện thoại": "dien_thoai",
    "dien thoai": "dien_thoai",
    "phone": "dien_thoai",
    "laptop": "laptop",
    "máy tính xách tay": "laptop",
    "may tinh xach tay": "laptop",
    "màn hình": "man_hinh",
    "man hinh": "man_hinh",
    "tablet": "may_tinh_bang",
    "máy tính bảng": "may_tinh_bang",
    "may tinh bang": "may_tinh_bang",
    "tai nghe": "tai_nghe_bluetooth",
    "bluetooth": "tai_nghe_bluetooth",
    "loa": "loa",
    "pc": "pc",
    "micro": "micro",
    "phụ kiện": "phu_kien",
    "phu kien": "phu_kien",
}


def normalize_text(text: object) -> str:
    text = str(text or "").lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def detect_category(question: str) -> str | None:
    q = normalize_text(question)
    for keyword, category in CATEGORY_KEYWORDS.items():
        if keyword in q:
            return category
    return None


def query_terms(question: str) -> list[str]:
    q = normalize_text(question)
    terms = re.findall(r"[\wÀ-ỹ]+", q)
    stopwords = {
        "tôi", "toi", "cần", "can", "tìm", "tim", "mua", "cho", "xin",
        "sản", "phẩm", "san", "pham", "nào", "nao", "có", "co", "không",
        "khong", "giá", "gia", "bao", "nhiêu", "nhieu", "tư", "vấn", "tu", "van",
    }
    return [term for term in terms if len(term) >= 2 and term not in stopwords]


def build_search_text(row: pd.Series) -> str:
    fields = [
        row.get("ten_san_pham", ""),
        row.get("thuong_hieu", ""),
        row.get("category", ""),
        row.get("mo_ta_ngan", ""),
        row.get("specs_text", ""),
    ]
    return normalize_text(" ".join(str(field) for field in fields))


products = products.copy()
products["search_text"] = products.apply(build_search_text, axis=1)


def search_products(question: str, top_k: int = TOP_K_PRODUCTS) -> pd.DataFrame:
    category = detect_category(question)
    terms = query_terms(question)

    candidates = products
    if category:
        candidates = candidates[candidates["category"].eq(category)]

    scored = []
    for idx, row in candidates.iterrows():
        text = row["search_text"]
        score = 0
        for term in terms:
            if term in text:
                score += 1
        if category:
            score += 2
        if score > 0:
            scored.append((score, idx))

    scored = sorted(scored, reverse=True)[:top_k]
    if not scored:
        return candidates.head(top_k).copy()
    return products.loc[[idx for _, idx in scored]].copy()


search_products("tìm điện thoại RAM 8GB", top_k=3)[["product_id", "ten_san_pham", "category"]]


## Build Product Context


In [ ]:
def compact_text(value: object, max_chars: int = 900) -> str:
    text = re.sub(r"\s+", " ", str(value or "")).strip()
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rsplit(" ", 1)[0] + "..."


def build_product_context(results: pd.DataFrame, max_chars: int = MAX_CONTEXT_CHARS) -> str:
    blocks = []
    for _, row in results.iterrows():
        block = f"""
product_id: {row.get('product_id', '')}
Tên: {row.get('ten_san_pham', '')}
Thương hiệu: {row.get('thuong_hieu', '')}
Danh mục: {row.get('category', '')}
Giá: {row.get('gia_ban', '') or 'Liên hệ'}
Mô tả: {compact_text(row.get('mo_ta_ngan', ''), 350)}
Thông số: {compact_text(row.get('specs_text', ''), 900)}
URL: {row.get('url', '')}
""".strip()
        blocks.append(block)

    context = "\n\n---\n\n".join(blocks)
    return context[:max_chars]


sample_results = search_products("tìm điện thoại RAM 8GB", top_k=3)
print(build_product_context(sample_results)[:1500])


## Load Base Model and LoRA Adapter


If you see `operator torchvision::nms does not exist` or `torchvision has no attribute extension`, `torchvision` is usually incompatible with your installed `torch`. The install cell above uninstalls `torchvision`; after running it, restart the kernel and rerun the notebook from the beginning.


In [ ]:
import os

os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["TRANSFORMERS_NO_VISION"] = "1"
os.environ["TRANSFORMERS_NO_AUDIO"] = "1"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

if torch.cuda.is_available():
    DEVICE = "cuda"
    TORCH_DTYPE = torch.float16
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    TORCH_DTYPE = torch.float16
else:
    DEVICE = "cpu"
    TORCH_DTYPE = torch.float32

print("Device:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=TORCH_DTYPE,
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
model.to(DEVICE)

print("Loaded model and adapter")


If you have an NVIDIA GPU and need lower VRAM usage, use the optional 4-bit loading cell below instead of the standard model-loading cell.


In [ ]:
SYSTEM_PROMPT = """
Bạn là chatbot tư vấn mua sắm. Chỉ trả lời dựa trên dữ liệu sản phẩm được cung cấp trong ngữ cảnh.
Nếu dữ liệu không có thông tin, hãy nói rõ là chưa có thông tin. Không bịa giá, tồn kho, khuyến mãi hoặc thông số.
Trả lời ngắn gọn, dễ hiểu, ưu tiên gợi ý sản phẩm kèm lý do và link nếu có.
""".strip()


def generate_answer(question: str, top_k: int = TOP_K_PRODUCTS, max_new_tokens: int = 384) -> str:
    results = search_products(question, top_k=top_k)
    context = build_product_context(results)

    user_prompt = f"""
Câu hỏi của khách hàng:
{question}

Dữ liệu sản phẩm liên quan:
{context}

Hãy trả lời khách hàng dựa trên dữ liệu trên.
""".strip()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


print(generate_answer("Tìm điện thoại RAM 8GB", top_k=5))


## Chat Function


In [ ]:
while True:
    question = input("Bạn: ").strip()
    if question.lower() in {"q", "quit", "exit", "thoát", "thoat"}:
        break
    answer = generate_answer(question)
    print("Bot:", answer)
    print()
